# 2 - Angle tag applier.
This notebook implements the model deveoloped in the `/notebooks/machine learning notebooks/2 - Angle classifier.ipynb` notebook. The outcome of that notebook was that both ways of training the data will be used, with a hard voting being done at the end - model that outputs the highest certainty is the winner. 

Since notebook 1 showed us that sometimes an image is usable, sometimes it isn't depending on the chosen model, I'll run both models against the full dataset of images. This has one major advantage: if I improve the bintagger alghorithm, I already have an angle tagger. It could also be interesting to see what the certainty scores are of predicted angles of 'crappy' images versus 'good' images; though the latter is an idea for further research and not the point of this exercise. 

The output of this data will be stored in SQL with four columns being used (two per model: tag and certainty) for storing the predictions and one column to reference the primary key of an image.

In [1]:
import pandas as pd
#from PIL import Image
#import matplotlib.pyplot as plt
import tensorflow as tf

import os
import sys
import numpy as np
import tensorflow as tf
import json

sys.path.append('../../utils')
import config_handling as conf
from database import Database
import file_io 
import implot
import cnn_helpers

2025-03-06 20:47:15.823214: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 2.1 Settings for this notebook

Setting 

In [2]:
CONFFILE = '../../config/automotive.conf.ini'
BATCH_SIZE = 1000
USE_BBOXES = True
SHAPE = 256

In [3]:
basedir, db  = conf.applyconf(CONFFILE)

Connection established


In [4]:
confdata = conf.read_config(CONFFILE)

In [5]:
table_query = """CREATE TABLE IF NOT EXISTS angletag_predicts (
    image_id INT PRIMARY KEY,
    model_label VARCHAR(32), 
    model_score FLOAT
);"""
db.execute_query(table_query)

[]

In [6]:
cnn_helpers.system_override()
device = cnn_helpers.system_pick_device()

System override applied - check if GPU is detected
Using GPU for deep learning.


2025-03-06 20:47:17.916524: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-06 20:47:20.291474: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-06 20:47:20.291542: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero


In [7]:
dirs = confdata['directories']
angle_model_dir = os.path.join(dirs['root_dir'], dirs['final_models_dirname'], dirs['final_angle_dir'])


angle_model = cnn_helpers.load_one_model(angle_model_dir)

2025-03-06 20:47:20.316379: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-06 20:47:20.316503: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-06 20:47:20.316553: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-06 20:47:20.316647: I external/local_xla/xla/stream_executor/rocm/rocm_executor.cc:920] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-03-06 20:47:20.316701: I external/local_xla/xla/stream_executor/rocm/rocm_executor.

In [8]:
def load_jsondump_to_encoder(dir, extension = '.json', key_to_int = True):
    files = [f for f in os.listdir(dir) if f.endswith(extension)]
    assert len(files) > 0
    file = files[0]
    filedir = os.path.join(dir, file)
    print(filedir)
    file = open(filedir, 'r', encoding='utf8')
    content = json.load(file)
    file.close()
    if key_to_int: 
        content = {int(k): v for k, v in content.items()}
    return content

labeldict = load_jsondump_to_encoder(angle_model_dir)

/home/frederic/Documents/automotive_project/final models/angles/encoded_labels.json


## Make the predictions: 
We'll be using a loop to predict

In [9]:
last_record_query = "SELECT MAX(image_id) as id FROM angletag_predicts"
last_record = db.execute_query(last_record_query)
record_id = int(last_record[0]['id']) if last_record[0]['id'] is not None else 0
print(f"Starting from {record_id} in batches of {BATCH_SIZE}")

Starting from 0 in batches of 1000


In [10]:
img_query = "SELECT * FROM images WHERE id > %s ORDER BY id ASC LIMIT %s;"
img_table = pd.DataFrame(db.execute_query(img_query, [record_id, BATCH_SIZE]))
while len(img_table) > 0:
    img_table = cnn_helpers.cast_bbox_values(img_table)
    img_table = file_io.make_absolute_path(img_table, basedir, 'image_path', 'abs_path', True)
    images = []
    ids = []
    for _, row in img_table.iterrows():
        record_id = row['id']
        image = row['abs_path']
        bboxvalues = [row['yolobox_top_left_x'],row['yolobox_top_left_y'],row['yolobox_bottom_right_x'],row['yolobox_bottom_right_y']]
        img_matrix = cnn_helpers.preprocess_image(image, USE_BBOXES, bboxvalues, SHAPE)
        images.append(img_matrix)
        ids.append(record_id)
    predictions = angle_model.predict(np.array(images), verbose=0)  #disable progress bar so it can print and override the record_id
    prediction_integers = np.argmax(predictions, axis=1) #get int labels of prediciton
    prediction_labels = [labeldict[i] for i in prediction_integers]  #get the string labels
    scores = np.max(predictions, axis=1)  #get the certainty score  
    img_table = pd.DataFrame(db.execute_query(img_query, [record_id, BATCH_SIZE]))
    db.start_transaction()
    for id, label, score in zip(ids, prediction_labels, scores):
        insert_query = "INSERT INTO angletag_predicts VALUES(%s, %s, %s)"
        insert_data = [id, label, score]
        db.execute_query(insert_query, insert_data)
    db.commit_transaction()

    print(record_id, end='\r', flush=True)    

    


I0000 00:00:1741290445.629208   72227 service.cc:146] XLA service 0x74e004106a20 initialized for platform ROCM (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1741290445.629244   72227 service.cc:154]   StreamExecutor device (0): AMD Radeon RX 6700 XT, AMDGPU ISA version: gfx1030
2025-03-06 20:47:25.634601: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1741290456.575636   72227 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


KeyboardInterrupt: 